In [ ]:
import torch
from torchvision import datasets
from torchvision.transforms import v2
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torch import nn

#### Preprocessing

In [ ]:
device = (
    "cuda" if torch.cuda.is_available()
    else "cpu"
)
print(f"Using Device: {device}")

In [ ]:
transformer = v2.Compose([
            v2.ToImage(),
            v2.ToDtype(torch.float32,scale=True),
            v2.RandomHorizontalFlip(0.5),
            v2.RandomRotation(15),
            v2.RandomHorizontalFlip(),
            v2.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))]
)

In [ ]:
train_data = datasets.CIFAR100(
        root = r"data",
        download = False,
        train = True,
        transform=transformer
)

In [ ]:
test_data = datasets.CIFAR100(
        root = r"data",
        download = False,
        train = False,
        transform=transformer
)

In [ ]:
train_dataloader = DataLoader(train_data,batch_size=64,shuffle=True)
test_dataloader = DataLoader(test_data,batch_size=64,shuffle=True)

In [ ]:
print(len(train_data))
print(len(test_data))

In [ ]:
labels = train_data.targets
test_labels = test_data.targets

In [ ]:
print(torch.tensor(labels).bincount())
print(torch.tensor(labels).unique())
print(torch.tensor(labels).bincount().size())
print()
print(torch.tensor(test_labels).bincount())
print(torch.tensor(test_labels).unique())
print(torch.tensor(test_labels).bincount().size())

In [ ]:
images = train_data.data
test_images = test_data.data

In [ ]:
print(images.shape)
print(test_images.shape)

In [ ]:
print(train_data[0][0].shape)
print(test_data[0][0].shape)

In [ ]:
uniq_sizes = set()
for img in images:
    uniq_sizes.add(img.shape)
for img in test_images:
    uniq_sizes.add(img.shape)
if (len(uniq_sizes))==1:
    print("Uniform size")
    print(uniq_sizes.pop())
else:
    print("Different sized images present")

In [ ]:
i = torch.randint(len(images),(1,))
plt.imshow(images[i],interpolation="bicubic")
print(images[i].shape)

#### NeuralNetwork

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.sequential_stack = nn.Sequential(
            nn.Conv2d(3,32,3,padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32,64,3,padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Dropout2d(0.25),
            
            nn.Conv2d(64,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128,128,3,padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Dropout2d(0.25),

            nn.Conv2d(128,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256,256,3,padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Dropout2d(0.25),

            nn.Conv2d(256,512,3,padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.Conv2d(512,512,3,padding=1),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Dropout2d(0.25),
            
            nn.Flatten(),
            nn.Linear(2*2*512,800),
            nn.ReLU(),
            nn.Linear(800,800),
            nn.ReLU(),
            nn.Dropout1d(0.5),
            nn.Linear(800,100)
        )
    def forward(self,X):
        logits = self.sequential_stack(X)
        return logits

In [ ]:
model = NeuralNetwork().to(device)
print(model)

In [ ]:
epochs = 80
learning_rate = 0.01
batch_size = 64

In [ ]:
loss_fn = nn.CrossEntropyLoss()

In [ ]:
optimizer = torch.optim.SGD(model.parameters(),lr=learning_rate,momentum=0.9)

In [ ]:
scheduler = torch.optim.lr_scheduler.StepLR(optimizer,step_size=10,gamma=0.5)

In [ ]:
def train_loop(dataloader,model,loss_fn,optimizer):
    size = len(dataloader.dataset)
    for batch, (x,y) in enumerate(dataloader):
        x,y = x.to(device),y.to(device)
        
        pred = model(x)
        loss = loss_fn(pred,y)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if (batch%100==0):
            loss = loss.item()
            curr = batch*batch_size+len(x)
            print(f"loss: {loss:>7f} at datapoint [{curr:>5d}/{size:>5d}]")

In [ ]:
def test_loop(dataloader,model,loss_fn):
    model.eval()
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    test_loss, correct = 0, 0
    with torch.no_grad():
        for x,y in dataloader:
            x,y = x.to(device),y.to(device)
            
            pred = model(x)
            test_loss += loss_fn(pred,y).item()
            correct += (pred.argmax(1)==y).type(dtype=torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error:\nAccuracy: {100*correct:>4f}%, Avg loss: {test_loss:>8f}")

In [ ]:
for i in range(epochs):
    print(f"Epoch {i+1}\n--------------------------")
    train_loop(train_dataloader,model,loss_fn,optimizer)
    test_loop(test_dataloader,model,loss_fn)
    scheduler.step()
print("Done")

In [ ]:
# Accuracy: 56.150000%, Avg loss: 4.096072